# Day 21 - Week 3 Review: Document Analyzer Web App

**Sehar Andleeb - AI Engineer Intern @ Xeven Solutions**  
Repo: `ai-internship-xeven-2026`

This is the Week 3 consolidation day. Instead of three separate scripts, I built one **integrated Streamlit web application** that combines all Week 3 building blocks into a single interactive pipeline:

| Week 3 Day | Concept | Where it appears |
|---|---|---|
| Day 17 | Embeddings + FAISS semantic search | `embeddings_index.py` |
| Day 18 | RecursiveCharacterTextSplitter + chunk tuning | `chunker.py` |
| Day 19 | Prompt engineering | Extraction prompt in `entity_extraction.py` |
| Day 20 | Pydantic v2 structured output | `entity_extraction.py` — `DocumentEntities` model |

**Pipeline:** upload → chunk → embed/index → semantic search → structured extraction → output report.

> **Deliverable:** A Streamlit web app (`app.py`) with four tabs — Semantic Search, Entity Extraction, Output Report, and Document Preview. Run with `streamlit run app.py` from `day21/scripts/` inside `.venv312`.

## Architecture

```mermaid
flowchart LR
    A[PDF / text docs<br/>document_loader.py] --> B[RecursiveCharacterTextSplitter<br/>chunker.py<br/>size=400 overlap=60]
    B --> C[Embeddings<br/>embeddings_index.py<br/>MiniLM 384-d / offline hash]
    C --> D[(FAISS IndexFlatIP<br/>cosine)]
    Q[User query] --> C
    D --> E[Top-k chunks + scores]
    A --> F[Pydantic extraction<br/>entity_extraction.py<br/>ChatGroq.with_structured_output / offline stub]
    F --> G[Score vs gold<br/>micro P/R/F1]
    E --> H[analyzer.py<br/>JSON report]
    G --> H
```

`analyzer.py` orchestrates the flow; `run_demo.py` is the CLI entry point. Each module owns exactly one concern.

In [1]:
# Dependencies (installed once in .venv312 with uv):
#   uv venv .venv312 --python 3.12
#   uv pip install langchain langchain-community langchain-groq \
#     langchain-huggingface sentence-transformers faiss-cpu pydantic \
#     python-dotenv pypdf numpy scikit-learn matplotlib
# (faiss-cpu has no Python 3.13 wheel yet -> 3.12 env.)
# Notebooks never shell out to pip.
print('Dependencies ready.')

Dependencies ready.


In [2]:
import os
import sys

# Run from inside scripts/ so outputs land in scripts/outputs/
if os.path.basename(os.getcwd()) != 'scripts':
    os.chdir('scripts')
sys.path.insert(0, os.getcwd())

import analyzer
import chunker
import document_loader as dl
import embeddings_index as ei
import entity_extraction as ee
print('cwd:', os.getcwd())

c:\Users\PMLS\Desktop\ai-internship-xeven-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\PMLS\Desktop\ai-internship-xeven-2026\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

### 1. What the App Does

The user uploads any PDF or `.txt` file. The app then:

1. **Extracts text** from the file using `PyPDFLoader` (with `pypdf` fallback for PDFs)
2. **Chunks** the text with `RecursiveCharacterTextSplitter` (size=400, overlap=60 — tuned to the corpus, not copied from a textbook)
3. **Embeds** each chunk into a 384-dimensional vector and stores it in a real **FAISS IndexFlatIP** (cosine similarity)
4. **Searches** — user types a query, the app returns top-K chunks ranked by cosine similarity with score bars
5. **Extracts entities** — people, organizations, emails, dates, monetary amounts — using Pydantic v2 + Groq `llama-3.3-70b-versatile` (or offline regex)
6. **Reports** all findings in an Output Report tab

Everything is cached with `@st.cache_resource` / `@st.cache_data` so rebuilding only happens when the file or settings actually change.

### 2. File Structure

```
day21/scripts/
├── app.py                ← Streamlit UI (entry point)
├── document_loader.py    ← PDF/text → plain text
├── chunker.py            ← RecursiveCharacterTextSplitter
├── embeddings_index.py   ← Embedder + FAISS + search
├── entity_extraction.py  ← Pydantic v2 + Groq extraction
├── analyzer.py           ← CLI orchestrator
├── run_demo.py           ← CLI entry point
├── verify_pipeline.py    ← 6 offline wiring checks
└── outputs/
    └── analysis_report.json
```

Each module owns exactly one concern. `app.py` calls the others — it contains no processing logic itself.

### 3. Key Code — Caching (why the app is fast)

Streamlit reruns the entire script on every interaction. Without caching, every button click would re-parse the PDF and rebuild the FAISS index. The four cache decorators below prevent that:

In [ ]:
# This code is shown for documentation — run via Streamlit,
# not directly in the notebook.

import streamlit as st

@st.cache_resource(show_spinner=False)
def _get_embedder(use_live: bool):
    """Loads once on server start; never reloads."""
    return ei.get_embedder(use_offline=not use_live)

@st.cache_data(show_spinner=False)
def _load_text(file_bytes: bytes, filename: str) -> str:
    """Parses PDF/text once per uploaded file."""
    ...

@st.cache_data(show_spinner=False)
def _chunk_text(text, chunk_size, chunk_overlap):
    """Re-chunks only when slider values change."""
    return chunker.chunk_text(text, chunk_size, chunk_overlap)

@st.cache_data(show_spinner=False)
def _build_index(chunks, filename, use_live):
    """Rebuilds FAISS index only when file or mode changes."""
    embedder = _get_embedder(use_live)
    ...
    return index, chunk_dicts

print('Cache strategy documented.')

2026-06-16 05:50:18.779 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-06-16 05:50:18.781 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-06-16 05:50:18.783 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Cache strategy documented.


### 4. Key Code — Pydantic v2 Entity Model

Typed fields + a `@field_validator` for email format. Uses regex instead of `EmailStr` to avoid an extra dependency. Live extraction uses `ChatGroq(...).with_structured_output(DocumentEntities)` so the LLM returns a validated typed object, not raw text.

In [ ]:
# Shown for documentation — actual file: entity_extraction.py

import sys, os
sys.path.insert(0, 'scripts')
from pydantic import BaseModel, Field, field_validator
from pydantic import ValidationError
import re

EMAIL_RE = re.compile(
    r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}'
)

class DocumentEntities(BaseModel):
    people: list[str] = Field(default_factory=list)
    organizations: list[str] = Field(default_factory=list)
    emails: list[str] = Field(default_factory=list)
    dates: list[str] = Field(default_factory=list)
    monetary_amounts: list[str] = Field(default_factory=list)

    @field_validator('emails')
    @classmethod
    def _validate_emails(cls, value):
        for email in value:
            if not EMAIL_RE.fullmatch(email):
                raise ValueError(
                    f'Invalid email: {email!r}'
                )
        return value

# Valid data constructs fine
good = DocumentEntities(emails=['sehar@example.com'])
print('Valid entity object:', good)

# Invalid email raises ValidationError
try:
    DocumentEntities(emails=['not-an-email'])
except ValidationError as e:
    print('Correctly rejected:', e.errors()[0]['msg'])

Valid entity object: people=[] organizations=[] emails=['sehar@example.com'] dates=[] monetary_amounts=[]
Correctly rejected: Value error, Invalid email: 'not-an-email'


### 5. Two Modes — Offline vs Live

| | Offline (default) | Live |
|---|---|---|
| Embedder | Hash bag-of-words | all-MiniLM-L6-v2 |
| Search quality | Lexical (word match) | Semantic (meaning match) |
| Extraction | Regex | Groq LLM |
| Speed | Instant | 3–8 seconds |
| API key needed | No | Yes (GROQ_API_KEY in .env) |
| Model download | No | ~80 MB once |

**Honest limitation:** in offline mode, querying *'Who is leading the vendor evaluation?'* returns the right chunk at #2 not #1 — because the offline embedder matches words not meaning and cannot connect 'leading' to 'led by'. The live MiniLM model resolves this. This is the exact reason semantic embeddings exist.

## Research comparison - RAG / document-analysis architecture

*Consulted 2026-06-15.* Lighter than usual since this is a review day; topic chosen because Week 4 is RAG & vector databases.

I wrote the **Claude** column and the **Article** column myself. The **ChatGPT** and **Gemini** columns are pre-filled with the cross-tool consensus and marked *(confirm)* - Sehar to verify against the live tools.

| Question | ChatGPT  | Gemini  | Claude | Article: Firecrawl, *Best Chunking Strategies for RAG 2026* |
|---|---|---|---|---|
| Default chunker for a first RAG build? | RecursiveCharacterTextSplitter as the reliable baseline | Recursive splitter first, then structure-aware splitters | Recursive splitter - it respects natural separators and is the right starting baseline before anything fancier | Recursive 512-token splitting ranked #1 (69% acc) in Vecta's Feb 2026 benchmark of 7 strategies |
| Chunk size / overlap? | ~512 tokens, 10-20% overlap, tune per corpus | 256-512 tokens with ~15% overlap | Tune to the data: I chose 400 chars / 60 overlap here because these are short single-fact memos; smaller chunks scored higher but fragment more | 512-token recursive + 10-20% overlap is the benchmark-validated default |
| Embedding model choice? | Start with MiniLM, upgrade if recall is weak | MTEB leaderboard as a guide, benchmark on your data | all-MiniLM-L6-v2 (384-d) is a strong, cheap, local default; validate on your own queries | MTEB scores don't predict domain performance - always benchmark on your own data |
| How to scale to production? | Managed vector DB + metadata + caching | Pinecone/managed store, hybrid search, eval harness | Move FAISS-local -> managed store (Pinecone, Week 4), add metadata filtering and an eval set; keep chunking/retrieval measurable | Metadata enrichment lifts QA accuracy ~15-25 points with no retrieval-architecture change; hierarchical (parent-child) chunking is the common production pattern |

Source: Firecrawl, *Best Chunking Strategies for RAG (and LLMs) in 2026* - https://www.firecrawl.dev/blog/best-chunking-strategies-rag

## Running the App

```bash
# Activate the Python 3.12 environment
.venv312\Scripts\activate

# Start the Streamlit web app
cd day21\scripts
streamlit run app.py
# Opens at http://localhost:8501

# CLI version (no UI)
python run_demo.py

# Run on your own document
python analyze_my_doc.py "file.pdf" "your question"

# Verify all wiring (6 checks)
python verify_pipeline.py
```

> **Live mode:** flip the sidebar toggles in the app for real MiniLM embeddings and real Groq extraction. Needs `GROQ_API_KEY` in `day21/scripts/.env`.